# BuzzSpot - Train+Val Model A + Model B Ensemble

This notebook does **not train** and does **not tune on validation**.

It uses the two final-fit checkpoints trained on official `train + valid`:

- **Model A:** full-frame YOLO26m
- **Model B:** sliced YOLO26m

It applies the exact historically validated ensemble recipe from the earlier clean validation experiment:

- Model A contributes all classes
- Model B contributes bee, bumblebee, and moth
- Model B hoverfly predictions are excluded
- `A_THR = 0.001`
- `B_THR = 0.001`
- `B_SLICE_IOS = 0.50`
- `B_SCORE_SCALE = 0.75`
- final class-aware NMS IoU = `0.60`


## 1. Install

In [3]:
!pip install -q ultralytics tqdm

## 2. Mount Drive and locate the current test-phase archive

In [4]:
from google.colab import drive
drive.mount("/content/drive")

import glob
from pathlib import Path

tar_hits = glob.glob(
    "/content/drive/MyDrive/**/19557529600-BuzzSet_challenge_testphase.tar",
    recursive=True,
)
if not tar_hits:
    tar_hits = glob.glob(
        "/content/drive/MyDrive/**/*BuzzSet_challenge_testphase*.tar*",
        recursive=True,
    )

assert tar_hits, "BuzzSpot test-phase tar file not found under MyDrive"
TAR_PATH = Path(tar_hits[0])

print("test-phase archive:", TAR_PATH)


Mounted at /content/drive
test-phase archive: /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/19557529600-BuzzSet_challenge_testphase.tar


## 3. Configuration and final-fit checkpoints

In [5]:
from pathlib import Path
import shutil
import json
import gc
import zipfile
import tarfile
from collections import defaultdict, Counter

CLASS_NAMES = {
    1: "bee",
    2: "bumblebee",
    3: "hoverfly",
    4: "moth",
}

PROJECT_DIR = Path("/content/drive/MyDrive/BuzzSpot")
WEIGHTS_DIR = PROJECT_DIR / "weights"
SUBMISSIONS_DIR = PROJECT_DIR / "submissions"
ENSEMBLE_DIR = PROJECT_DIR / "ensemble_ab_trainval"

SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_A_TAG = "yolo26m_32ep_rare_os_cm5_trainval_finalfit"
MODEL_B_TAG = "yolo26m_sliced700_trainval_from_fullframeA_12ep"
ENSEMBLE_TAG = f"{MODEL_A_TAG}_plus_{MODEL_B_TAG}"

# Exact final-fit checkpoints used for the new ensemble.
MODEL_A_DRIVE_WEIGHTS = WEIGHTS_DIR / "yolo26m_32ep_rare_os_cm5_trainval_finalfit_last.pt"
MODEL_B_DRIVE_WEIGHTS = WEIGHTS_DIR / "yolo26m_sliced700_trainval_from_fullframeA_12ep_last.pt"

LOCAL_MODEL_A_WEIGHTS = Path("/content/modelA_trainval_last.pt")
LOCAL_MODEL_B_WEIGHTS = Path("/content/modelB_trainval_last.pt")

assert MODEL_A_DRIVE_WEIGHTS.exists(), f"Missing Model A weights: {MODEL_A_DRIVE_WEIGHTS}"
assert MODEL_B_DRIVE_WEIGHTS.exists(), f"Missing Model B weights: {MODEL_B_DRIVE_WEIGHTS}"
assert MODEL_A_DRIVE_WEIGHTS.name.endswith("_last.pt")
assert MODEL_B_DRIVE_WEIGHTS.name.endswith("_last.pt")

shutil.copy2(MODEL_A_DRIVE_WEIGHTS, LOCAL_MODEL_A_WEIGHTS)
shutil.copy2(MODEL_B_DRIVE_WEIGHTS, LOCAL_MODEL_B_WEIGHTS)

print("Model A:", MODEL_A_DRIVE_WEIGHTS)
print("Model B:", MODEL_B_DRIVE_WEIGHTS)
print("Local A:", LOCAL_MODEL_A_WEIGHTS)
print("Local B:", LOCAL_MODEL_B_WEIGHTS)

LOCAL_DATA_DIR = Path("/content/buzz_ensemble_ab_trainval")
TEST_IMAGE_DIR = LOCAL_DATA_DIR / "images" / "test_testphase"
ANNOT_DIR = LOCAL_DATA_DIR / "annotations"
PRED_DIR = LOCAL_DATA_DIR / "predictions"

for d in [TEST_IMAGE_DIR, ANNOT_DIR, PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Low candidate thresholds preserve ranked detections for final composition.
A_CANDIDATE_CONF = 0.001
B_CANDIDATE_CONF = 0.001

# Model A full-frame inference.
A_IMGSZ = 1536
A_IOU = 0.60
A_MAX_DET = 100
A_INFER_BATCH = 2
A_AUGMENT = False

# Model B sliced inference.
SLICE_SIZE = 700
OVERLAP = 0.20
STRIDE = int(SLICE_SIZE * (1.0 - OVERLAP))  # 560
B_IMGSZ = 704
B_IOU = 0.45
B_MAX_DET_PER_SLICE = 100
B_INFER_BATCH_SLICES = 8
B_AUGMENT = False

MAX_DET_FINAL = 100

TEST_A_RAW_JSON = PRED_DIR / f"test_raw_{MODEL_A_TAG}_conf001.json"
TEST_B_RAW_JSON = PRED_DIR / f"test_raw_{MODEL_B_TAG}_sliced_conf001.json"

FORCE_RERUN_TEST_INFERENCE = False

print("A full-frame settings:", A_IMGSZ, A_IOU, A_MAX_DET, A_INFER_BATCH)
print("B sliced settings:", SLICE_SIZE, OVERLAP, STRIDE, B_IMGSZ, B_IOU)
print("raw A cache:", TEST_A_RAW_JSON)
print("raw B cache:", TEST_B_RAW_JSON)


Model A: /content/drive/MyDrive/BuzzSpot/weights/yolo26m_32ep_rare_os_cm5_trainval_finalfit_last.pt
Model B: /content/drive/MyDrive/BuzzSpot/weights/yolo26m_sliced700_trainval_from_fullframeA_12ep_last.pt
Local A: /content/modelA_trainval_last.pt
Local B: /content/modelB_trainval_last.pt
A full-frame settings: 1536 0.6 100 2
B sliced settings: 700 0.2 560 704 0.45
raw A cache: /content/buzz_ensemble_ab_trainval/predictions/test_raw_yolo26m_32ep_rare_os_cm5_trainval_finalfit_conf001.json
raw B cache: /content/buzz_ensemble_ab_trainval/predictions/test_raw_yolo26m_sliced700_trainval_from_fullframeA_12ep_sliced_conf001.json


## 4. Load the current test annotation and extract only submission keyframes

In [6]:
import json
import tarfile
import shutil
from pathlib import Path
from tqdm.auto import tqdm

if TEST_IMAGE_DIR.exists():
    shutil.rmtree(TEST_IMAGE_DIR)
TEST_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
ANNOT_DIR.mkdir(parents=True, exist_ok=True)

tf = tarfile.open(TAR_PATH, "r:*")
tar_members = {m.name: m for m in tf.getmembers() if m.isfile()}
tar_names = set(tar_members.keys())

def find_tar(rel):
    rel = rel.lstrip("/")
    candidates = (
        rel,
        f"BuzzSet_challenge/{rel}",
        f"BuzzSet_challenge_testphase/{rel}",
    )
    for cand in candidates:
        if cand in tar_names:
            return cand

    suffix = "/" + rel
    hits = [name for name in tar_names if name.endswith(suffix)]
    return hits[0] if hits else None

def load_tar_json(fname):
    member_name = find_tar(f"annotations/{fname}") or find_tar(fname)
    assert member_name is not None, f"{fname} not found in test-phase archive"

    with tf.extractfile(tar_members[member_name]) as f:
        obj = json.load(f)

    out = ANNOT_DIR / fname
    out.write_text(json.dumps(obj))
    print("loaded:", fname, "from", member_name)
    return obj

def flat_name(file_name):
    return file_name.replace("/", "__")

def extract_test_keyframes(test_coco):
    keyframes = [
        im for im in test_coco["images"]
        if im.get("is_keyframe") in [True, 1, "true", "True"]
    ]

    records = []
    seen_ids = set()

    for im in tqdm(keyframes, desc="extract test_testphase keyframes"):
        image_id = int(im["id"])
        assert image_id not in seen_ids, f"Duplicate test image id: {image_id}"
        seen_ids.add(image_id)

        member_name = (
            find_tar(f"test_testphase/{im['file_name']}")
            or find_tar(im["file_name"])
        )
        assert member_name is not None, f"Missing test image: {im['file_name']}"

        out_name = flat_name(im["file_name"])
        dst = TEST_IMAGE_DIR / out_name

        with tf.extractfile(tar_members[member_name]) as src, open(dst, "wb") as out:
            shutil.copyfileobj(src, out)

        records.append({
            "id": image_id,
            "file_name": im["file_name"],
            "flat_name": out_name,
            "path": str(dst),
            "width": int(im.get("width", 1920)),
            "height": int(im.get("height", 1080)),
        })

    records.sort(key=lambda x: x["id"])
    return records

test_coco = load_tar_json("test_testphase.json")
test_records = extract_test_keyframes(test_coco)
tf.close()

TEST_JSON_PATH = ANNOT_DIR / "test_testphase.json"

print("test keyframes:", len(test_records))
print("first record:", test_records[0])

assert len(test_records) == 4763, f"Expected 4763 test keyframes, got {len(test_records)}"
assert len({rec["id"] for rec in test_records}) == 4763


loaded: test_testphase.json from BuzzSpot_testphase/annotations/test_testphase.json


extract test_testphase keyframes:   0%|          | 0/4763 [00:00<?, ?it/s]

test keyframes: 4763
first record: {'id': 6, 'file_name': 'test_video_1/test_video_1_005730.jpg', 'flat_name': 'test_video_1__test_video_1_005730.jpg', 'path': '/content/buzz_ensemble_ab_trainval/images/test_testphase/test_video_1__test_video_1_005730.jpg', 'width': 1920, 'height': 1080}


## 5. Shared geometry and merge utilities

In [7]:
import math
from collections import defaultdict
import numpy as np

def slice_starts(length, size=SLICE_SIZE, stride=STRIDE):
    if length <= size:
        return [0]

    starts = list(range(0, length - size + 1, stride))
    last = length - size

    if starts[-1] != last:
        starts.append(last)

    return sorted(set(starts))

def xyxy_area(box):
    x1, y1, x2, y2 = box
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)

def intersection_area(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    return max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)

def iou_xyxy(a, b):
    inter = intersection_area(a, b)
    union = xyxy_area(a) + xyxy_area(b) - inter
    return 0.0 if union <= 0 else inter / union

def ios_xyxy(a, b):
    inter = intersection_area(a, b)
    denom = min(xyxy_area(a), xyxy_area(b))
    return 0.0 if denom <= 0 else inter / denom

def clip_xyxy(x1, y1, x2, y2, width, height):
    x1 = max(0.0, min(float(x1), width - 1))
    y1 = max(0.0, min(float(y1), height - 1))
    x2 = max(0.0, min(float(x2), width))
    y2 = max(0.0, min(float(y2), height))
    return x1, y1, x2, y2

def weighted_merge_cluster(cluster):
    scores = np.array([float(p["score"]) for p in cluster], dtype=np.float64)
    boxes = np.array([p["bbox_xyxy"] for p in cluster], dtype=np.float64)

    if scores.sum() <= 0:
        weights = np.ones_like(scores) / len(scores)
    else:
        weights = scores / scores.sum()

    merged_box = (boxes * weights[:, None]).sum(axis=0).tolist()
    merged_score = float(scores.max())

    out = cluster[0].copy()
    out["bbox_xyxy"] = merged_box
    out["score"] = merged_score
    return out

def merge_greedy_nmm_ios(preds, ios_thr=0.50, max_det=MAX_DET_FINAL):
    final = []

    for category_id in sorted({int(p["category_id"]) for p in preds}):
        cls_preds = [
            p for p in preds
            if int(p["category_id"]) == category_id
        ]
        cls_preds.sort(key=lambda p: float(p["score"]), reverse=True)

        while cls_preds:
            top = cls_preds[0]
            cluster = [top]
            rest = []

            for pred in cls_preds[1:]:
                if ios_xyxy(top["bbox_xyxy"], pred["bbox_xyxy"]) >= ios_thr:
                    cluster.append(pred)
                else:
                    rest.append(pred)

            final.append(weighted_merge_cluster(cluster))
            cls_preds = rest

    final.sort(key=lambda p: float(p["score"]), reverse=True)
    return final[:max_det]

def merge_class_aware_nms_iou(preds, iou_thr=0.60, max_det=MAX_DET_FINAL):
    final = []

    for category_id in sorted({int(p["category_id"]) for p in preds}):
        cls_preds = [
            p for p in preds
            if int(p["category_id"]) == category_id
        ]
        cls_preds.sort(key=lambda p: float(p["score"]), reverse=True)

        kept = []
        while cls_preds:
            top = cls_preds.pop(0)
            kept.append(top)
            cls_preds = [
                p for p in cls_preds
                if iou_xyxy(top["bbox_xyxy"], p["bbox_xyxy"]) < iou_thr
            ]

        final.extend(kept)

    final.sort(key=lambda p: float(p["score"]), reverse=True)
    return final[:max_det]

def scale_score(pred, scale=1.0):
    out = pred.copy()
    out["score"] = float(
        min(0.99999, max(0.0, float(out["score"]) * float(scale)))
    )
    return out

def filter_preds(preds, threshold=0.001, score_scale=1.0):
    return [
        scale_score(pred, score_scale)
        for pred in preds
        if float(pred["score"]) >= float(threshold)
    ]

def filter_to_categories(preds, keep_categories):
    keep_categories = {int(x) for x in keep_categories}
    return [
        pred for pred in preds
        if int(pred["category_id"]) in keep_categories
    ]

def group_by_image(preds):
    grouped = defaultdict(list)
    for pred in preds:
        grouped[int(pred["image_id"])].append(pred)
    return grouped

def raw_to_coco(preds):
    output = []

    for pred in preds:
        x1, y1, x2, y2 = pred["bbox_xyxy"]
        width = max(0.0, x2 - x1)
        height = max(0.0, y2 - y1)

        if width < 1 or height < 1:
            continue

        output.append({
            "image_id": int(pred["image_id"]),
            "category_id": int(pred["category_id"]),
            "bbox": [
                round(float(x1), 2),
                round(float(y1), 2),
                round(float(width), 2),
                round(float(height), 2),
            ],
            "score": round(float(pred["score"]), 5),
        })

    return output


## 6. Load both models and define test inference

In [8]:
import json
import gc
import torch
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
from ultralytics import YOLO

model_a = YOLO(str(LOCAL_MODEL_A_WEIGHTS))
model_b = YOLO(str(LOCAL_MODEL_B_WEIGHTS))

print("Model A classes:", model_a.names)
print("Model B classes:", model_b.names)

assert len(model_a.names) == 4
assert len(model_b.names) == 4

def predict_model_a_fullframe(records, out_json, force=False):
    out_json = Path(out_json)

    if out_json.exists() and not force:
        print("Loading cached Model A predictions:", out_json)
        return json.loads(out_json.read_text())

    raw_preds = []

    for start in tqdm(
        range(0, len(records), A_INFER_BATCH),
        desc="Model A full-frame test",
    ):
        batch_records = records[start:start + A_INFER_BATCH]
        paths = [record["path"] for record in batch_records]

        results = model_a.predict(
            source=paths,
            imgsz=A_IMGSZ,
            conf=A_CANDIDATE_CONF,
            iou=A_IOU,
            max_det=A_MAX_DET,
            batch=len(paths),
            augment=A_AUGMENT,
            agnostic_nms=False,
            verbose=False,
        )

        for record, result in zip(batch_records, results):
            width = int(record["width"])
            height = int(record["height"])

            if result.boxes is None or len(result.boxes) == 0:
                continue

            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                x1, y1, x2, y2 = clip_xyxy(
                    x1, y1, x2, y2, width, height
                )

                if x2 - x1 >= 1 and y2 - y1 >= 1:
                    raw_preds.append({
                        "image_id": int(record["id"]),
                        "category_id": int(box.cls[0]) + 1,
                        "bbox_xyxy": [x1, y1, x2, y2],
                        "score": float(box.conf[0]),
                    })

        if start % 100 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    out_json.write_text(json.dumps(raw_preds))
    print("Saved Model A raw predictions:", out_json)
    print("Model A raw detections:", len(raw_preds))
    print("Model A class counts:", dict(Counter(int(p["category_id"]) for p in raw_preds)))
    return raw_preds

def predict_model_b_sliced(records, out_json, force=False):
    out_json = Path(out_json)

    if out_json.exists() and not force:
        print("Loading cached Model B predictions:", out_json)
        return json.loads(out_json.read_text())

    all_raw = []

    def flush_batch(batch_images, jobs):
        if not batch_images:
            return []

        results = model_b.predict(
            source=batch_images,
            imgsz=B_IMGSZ,
            conf=B_CANDIDATE_CONF,
            iou=B_IOU,
            max_det=B_MAX_DET_PER_SLICE,
            batch=len(batch_images),
            augment=B_AUGMENT,
            agnostic_nms=False,
            verbose=False,
        )

        batch_raw = []

        for result, job in zip(results, jobs):
            image_id, offset_x, offset_y, width, height = job

            if result.boxes is None or len(result.boxes) == 0:
                continue

            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()

                x1 += offset_x
                x2 += offset_x
                y1 += offset_y
                y2 += offset_y

                x1, y1, x2, y2 = clip_xyxy(
                    x1, y1, x2, y2, width, height
                )

                if x2 - x1 >= 1 and y2 - y1 >= 1:
                    batch_raw.append({
                        "image_id": int(image_id),
                        "category_id": int(box.cls[0]) + 1,
                        "bbox_xyxy": [x1, y1, x2, y2],
                        "score": float(box.conf[0]),
                    })

        return batch_raw

    for index, record in enumerate(tqdm(records, desc="Model B sliced test")):
        image = Image.open(record["path"]).convert("RGB")

        width = int(record["width"])
        height = int(record["height"])

        x_starts = slice_starts(width)
        y_starts = slice_starts(height)

        batch_images = []
        jobs = []

        for offset_y in y_starts:
            for offset_x in x_starts:
                crop = image.crop((
                    offset_x,
                    offset_y,
                    offset_x + SLICE_SIZE,
                    offset_y + SLICE_SIZE,
                ))

                # Ultralytics NumPy inference expects OpenCV-style BGR arrays.
                crop_bgr = np.ascontiguousarray(
                    np.asarray(crop)[:, :, ::-1]
                )

                batch_images.append(crop_bgr)
                jobs.append((
                    int(record["id"]),
                    offset_x,
                    offset_y,
                    width,
                    height,
                ))

                if len(batch_images) == B_INFER_BATCH_SLICES:
                    all_raw.extend(flush_batch(batch_images, jobs))
                    batch_images = []
                    jobs = []

        if batch_images:
            all_raw.extend(flush_batch(batch_images, jobs))

        image.close()

        if index % 100 == 0:
            print(index, "/", len(records), "raw B detections:", len(all_raw))
            gc.collect()
            torch.cuda.empty_cache()

    out_json.write_text(json.dumps(all_raw))
    print("Saved Model B raw predictions:", out_json)
    print("Model B raw detections:", len(all_raw))
    print("Model B class counts:", dict(Counter(int(p["category_id"]) for p in all_raw)))
    return all_raw


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model A classes: {0: 'bee', 1: 'bumblebee', 2: 'hoverfly', 3: 'moth'}
Model B classes: {0: 'bee', 1: 'bumblebee', 2: 'hoverfly', 3: 'moth'}


## 7. Lock the historically validated ensemble recipe

In [9]:
FINAL_MODE = "A+B_gated_no_hoverfly"

FINAL_CONFIG = {
    "a_thr": 0.001,
    "b_thr": 0.001,
    "b_slice_ios": 0.50,
    "b_score_scale": 0.75,
    "final_method": "nms_iou",
    "final_thr": 0.60,
    "b_keep_name": "B_no_hoverfly",
    "b_keep_cats": [1, 2, 4],
}

# Hard guards prevent accidental deviation from the validated recipe.
assert FINAL_CONFIG["b_keep_cats"] == [1, 2, 4]
assert abs(FINAL_CONFIG["a_thr"] - 0.001) < 1e-12
assert abs(FINAL_CONFIG["b_thr"] - 0.001) < 1e-12
assert abs(FINAL_CONFIG["b_slice_ios"] - 0.50) < 1e-12
assert abs(FINAL_CONFIG["b_score_scale"] - 0.75) < 1e-12
assert FINAL_CONFIG["final_method"] == "nms_iou"
assert abs(FINAL_CONFIG["final_thr"] - 0.60) < 1e-12

config_path = ENSEMBLE_DIR / f"selected_config_{ENSEMBLE_TAG}.json"
config_path.write_text(json.dumps({
    "FINAL_MODE": FINAL_MODE,
    "FINAL_CONFIG": FINAL_CONFIG,
}, indent=2))

print("FINAL_MODE:", FINAL_MODE)
print(json.dumps(FINAL_CONFIG, indent=2))
print("saved config:", config_path)


FINAL_MODE: A+B_gated_no_hoverfly
{
  "a_thr": 0.001,
  "b_thr": 0.001,
  "b_slice_ios": 0.5,
  "b_score_scale": 0.75,
  "final_method": "nms_iou",
  "final_thr": 0.6,
  "b_keep_name": "B_no_hoverfly",
  "b_keep_cats": [
    1,
    2,
    4
  ]
}
saved config: /content/drive/MyDrive/BuzzSpot/ensemble_ab_trainval/selected_config_yolo26m_32ep_rare_os_cm5_trainval_finalfit_plus_yolo26m_sliced700_trainval_from_fullframeA_12ep.json


## 8. Run or load raw test predictions

In [10]:
test_a_raw = predict_model_a_fullframe(
    test_records,
    TEST_A_RAW_JSON,
    force=FORCE_RERUN_TEST_INFERENCE,
)

test_b_raw = predict_model_b_sliced(
    test_records,
    TEST_B_RAW_JSON,
    force=FORCE_RERUN_TEST_INFERENCE,
)

print("A raw predictions:", len(test_a_raw))
print("B raw predictions:", len(test_b_raw))
print("A raw predictions/image:", len(test_a_raw) / len(test_records))
print("B raw predictions/image:", len(test_b_raw) / len(test_records))


Model A full-frame test:   0%|          | 0/2382 [00:00<?, ?it/s]

Saved Model A raw predictions: /content/buzz_ensemble_ab_trainval/predictions/test_raw_yolo26m_32ep_rare_os_cm5_trainval_finalfit_conf001.json
Model A raw detections: 45883
Model A class counts: {1: 26209, 3: 17651, 2: 1219, 4: 804}


Model B sliced test:   0%|          | 0/4763 [00:00<?, ?it/s]

0 / 4763 raw B detections: 22
100 / 4763 raw B detections: 3450
200 / 4763 raw B detections: 4311
300 / 4763 raw B detections: 5266
400 / 4763 raw B detections: 6038
500 / 4763 raw B detections: 6798
600 / 4763 raw B detections: 7502
700 / 4763 raw B detections: 8392
800 / 4763 raw B detections: 9148
900 / 4763 raw B detections: 9937
1000 / 4763 raw B detections: 11641
1100 / 4763 raw B detections: 13424
1200 / 4763 raw B detections: 14795
1300 / 4763 raw B detections: 16312
1400 / 4763 raw B detections: 18847
1500 / 4763 raw B detections: 20961
1600 / 4763 raw B detections: 22981
1700 / 4763 raw B detections: 24620
1800 / 4763 raw B detections: 26318
1900 / 4763 raw B detections: 28599
2000 / 4763 raw B detections: 31841
2100 / 4763 raw B detections: 34430
2200 / 4763 raw B detections: 36389
2300 / 4763 raw B detections: 39647
2400 / 4763 raw B detections: 42751
2500 / 4763 raw B detections: 48554
2600 / 4763 raw B detections: 56317
2700 / 4763 raw B detections: 58187
2800 / 4763 raw 

## 9. Compose the fixed A+B ensemble submission

In [11]:
import json
import zipfile
import shutil
from pathlib import Path
from collections import Counter

test_a_by_image = group_by_image(test_a_raw)
test_b_by_image = group_by_image(test_b_raw)

def compose_fixed_ensemble(records):
    final = []

    for record in records:
        image_id = int(record["id"])

        # Model A owns all four classes.
        a_preds = filter_preds(
            test_a_by_image.get(image_id, []),
            threshold=FINAL_CONFIG["a_thr"],
            score_scale=1.0,
        )

        # Model B contributes only bee, bumblebee, and moth.
        b_raw = filter_preds(
            test_b_by_image.get(image_id, []),
            threshold=FINAL_CONFIG["b_thr"],
            score_scale=FINAL_CONFIG["b_score_scale"],
        )
        b_raw = filter_to_categories(
            b_raw,
            FINAL_CONFIG["b_keep_cats"],
        )

        # Merge duplicate detections created by overlapping Model B slices.
        b_merged = merge_greedy_nmm_ios(
            b_raw,
            ios_thr=FINAL_CONFIG["b_slice_ios"],
            max_det=MAX_DET_FINAL,
        )

        # Merge A and B using the historically validated final NMS.
        combined = a_preds + b_merged
        merged = merge_class_aware_nms_iou(
            combined,
            iou_thr=FINAL_CONFIG["final_thr"],
            max_det=MAX_DET_FINAL,
        )

        final.extend(merged)

    return raw_to_coco(final)

final_test_predictions = compose_fixed_ensemble(test_records)

mode_tag = "AplusB_gated_no_hoverfly_Bnohoverfly"
config_tag = "a0p001_b0p001_bm0p5_bscale0p75_fmnms_iou_0p6"

local_predictions_path = Path(
    f"/content/predictions_{ENSEMBLE_TAG}_{mode_tag}_{config_tag}.json"
)
local_zip_path = Path(
    f"/content/submission_{ENSEMBLE_TAG}_{mode_tag}_{config_tag}.zip"
)

drive_predictions_path = SUBMISSIONS_DIR / local_predictions_path.name
drive_zip_path = SUBMISSIONS_DIR / local_zip_path.name

local_predictions_path.write_text(json.dumps(final_test_predictions))

with zipfile.ZipFile(local_zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(local_predictions_path, arcname="predictions.json")

shutil.copy2(local_predictions_path, drive_predictions_path)
shutil.copy2(local_zip_path, drive_zip_path)

print("detections:", len(final_test_predictions))
print("predictions/image:", len(final_test_predictions) / len(test_records))
print("class counts:", {
    CLASS_NAMES[k]: v
    for k, v in sorted(Counter(
        int(p["category_id"]) for p in final_test_predictions
    ).items())
})
print("local predictions:", local_predictions_path)
print("local zip:", local_zip_path)
print("Drive predictions:", drive_predictions_path)
print("Drive zip:", drive_zip_path)


detections: 48750
predictions/image: 10.235145916439219
class counts: {'bee': 31432, 'bumblebee': 1971, 'hoverfly': 12833, 'moth': 2514}
local predictions: /content/predictions_yolo26m_32ep_rare_os_cm5_trainval_finalfit_plus_yolo26m_sliced700_trainval_from_fullframeA_12ep_AplusB_gated_no_hoverfly_Bnohoverfly_a0p001_b0p001_bm0p5_bscale0p75_fmnms_iou_0p6.json
local zip: /content/submission_yolo26m_32ep_rare_os_cm5_trainval_finalfit_plus_yolo26m_sliced700_trainval_from_fullframeA_12ep_AplusB_gated_no_hoverfly_Bnohoverfly_a0p001_b0p001_bm0p5_bscale0p75_fmnms_iou_0p6.zip
Drive predictions: /content/drive/MyDrive/BuzzSpot/submissions/predictions_yolo26m_32ep_rare_os_cm5_trainval_finalfit_plus_yolo26m_sliced700_trainval_from_fullframeA_12ep_AplusB_gated_no_hoverfly_Bnohoverfly_a0p001_b0p001_bm0p5_bscale0p75_fmnms_iou_0p6.json
Drive zip: /content/drive/MyDrive/BuzzSpot/submissions/submission_yolo26m_32ep_rare_os_cm5_trainval_finalfit_plus_yolo26m_sliced700_trainval_from_fullframeA_12ep_AplusB_

## 10. Strict submission validation

In [12]:
with zipfile.ZipFile(local_zip_path, "r") as archive:
    names = archive.namelist()

print("zip contents:", names)
assert names == ["predictions.json"], (
    f"Zip must contain exactly predictions.json, got {names}"
)

test_dimensions = {
    int(record["id"]): (
        int(record["width"]),
        int(record["height"]),
    )
    for record in test_records
}
valid_test_ids = set(test_dimensions)

outside_ids = [
    pred["image_id"]
    for pred in final_test_predictions
    if int(pred["image_id"]) not in valid_test_ids
]
assert not outside_ids, (
    "Predictions contain image IDs outside test keyframes: "
    f"{outside_ids[:10]}"
)

bad_categories = [
    pred for pred in final_test_predictions
    if int(pred["category_id"]) not in [1, 2, 3, 4]
]
assert not bad_categories, f"Invalid category IDs: {bad_categories[:5]}"

bad_scores = [
    pred for pred in final_test_predictions
    if not (0.0 <= float(pred["score"]) <= 1.0)
]
assert not bad_scores, f"Invalid scores: {bad_scores[:5]}"

degenerate_boxes = []
out_of_bounds_boxes = []

for pred in final_test_predictions:
    image_id = int(pred["image_id"])
    image_width, image_height = test_dimensions[image_id]
    x, y, width, height = map(float, pred["bbox"])

    if width <= 0 or height <= 0:
        degenerate_boxes.append(pred)
        continue

    if (
        x < 0
        or y < 0
        or x + width > image_width + 1e-6
        or y + height > image_height + 1e-6
    ):
        out_of_bounds_boxes.append(pred)

assert not degenerate_boxes, (
    f"Degenerate boxes found: {degenerate_boxes[:5]}"
)
assert not out_of_bounds_boxes, (
    f"Out-of-bounds boxes found: {out_of_bounds_boxes[:5]}"
)

predicted_image_ids = {
    int(pred["image_id"])
    for pred in final_test_predictions
}

print("submission checks passed")
print("predicted images:", len(predicted_image_ids), "/", len(test_records))
print("detections:", len(final_test_predictions))
print("ready to submit:", drive_zip_path)


zip contents: ['predictions.json']
submission checks passed
predicted images: 4741 / 4763
detections: 48750
ready to submit: /content/drive/MyDrive/BuzzSpot/submissions/submission_yolo26m_32ep_rare_os_cm5_trainval_finalfit_plus_yolo26m_sliced700_trainval_from_fullframeA_12ep_AplusB_gated_no_hoverfly_Bnohoverfly_a0p001_b0p001_bm0p5_bscale0p75_fmnms_iou_0p6.zip
